In [ ]:

import pandas as pd
from pathlib import Path

# =========================================================
# IMDb CROSS-SCALE COMPARISON
# =========================================================
# This notebook reproduces the same cross-scale comparison style
# used in the attached comparison_sfs notebook, but adapted to IMDb.
#
# IMPORTANT CHANGE:
# The corrected per-scale analysis now works at the granularity of:
#   (query_name, scale_label, run_phase)
# This is the correct granularity according to the paper.
#
# Because of that, this comparison notebook:
# 1) reads one analysis file per scale-factor result folder
# 2) filters by run_phase explicitly
# 3) keeps the same main output artifacts as the previous notebook
# 4) optionally also summarizes near-best preservation, because the
#    corrected individual analysis now computes it
#
# Expected input files (one per scale folder):
# - imdb_summary_hot_by_scale.csv
#   or
# - imdb_summary_hot.csv
#
# Main output artifacts:
# - imdb_cross_scale_schemalens_analysis_final.csv
# - imdb_cross_scale_summary_final.csv
# - imdb_cross_scale_secondary_winners_final.csv
# - imdb_cross_scale_control_winners_final.csv
#
# Optional extra artifact:
# - imdb_cross_scale_near_best_summary_final.csv
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------
base_results = Path(
    "/path/to/schemalens/imdb/results"
)

# Adjust these paths to your actual IMDb result folders
paths = {
    "sf0_25": base_results / "sf_025" / "imdb_analysis_outputs" / "imdb_summary_hot_by_scale.csv",
    "sf0_50":  base_results / "sf_050"  / "imdb_analysis_outputs" / "imdb_summary_hot_by_scale.csv",
    "sf_1":    base_results / "sf_1"    / "imdb_analysis_outputs" / "imdb_summary_hot_by_scale.csv",
}

# If your exported files use the older name, change here:
fallback_filename = "imdb_summary_hot.csv"

selected_run_phase = "hot"

# ---------------------------------------------------------
# 2) Read all per-scale analysis files
# ---------------------------------------------------------
dfs = []

for scale, path in paths.items():
    if not path.exists():
        fallback_path = path.parent / fallback_filename
        if fallback_path.exists():
            path = fallback_path
        else:
            raise FileNotFoundError(
                f"Could not find analysis file for {scale}.\n"
                f"Tried:\n- {path}\n- {fallback_path}"
            )

    df = pd.read_csv(path)

    # Ensure scale label exists and is correct
    if "scale_label" not in df.columns:
        df["scale_label"] = scale

    # Force expected scale in case the file contains only one-scale results
    if df["scale_label"].nunique() == 1:
        df["scale_label"] = scale

    # Keep only selected run phase if present
    if "run_phase" in df.columns:
        df = df[df["run_phase"] == selected_run_phase].copy()
    else:
        df["run_phase"] = selected_run_phase

    dfs.append(df)

cross_scale_schemalens_df = pd.concat(dfs, ignore_index=True)

# ---------------------------------------------------------
# 3) Build the main summary table
# Same style as the attached notebook
# ---------------------------------------------------------
agg_dict = dict(
    n_queries=("query_name", "nunique"),
    avg_DSR=("DSR", "mean"),
    top1_preservation=("top1_preserved_by_activated", "mean"),
    mean_activated_regret=("activated_regret", "mean"),
    mean_primary_regret=("primary_regret", lambda x: x.dropna().mean()),
    primary_winners=("best_group", lambda x: int((x == "primary").sum())),
    secondary_winners=("best_group", lambda x: int((x == "secondary_affected").sum())),
    control_winners=("best_group", lambda x: int((x == "control").sum())),
)

# If the corrected file contains near-best preservation, include it
if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    agg_dict["near_best_preservation"] = ("near_best_preserved_by_activated", "mean")

summary_df = (
    cross_scale_schemalens_df
    .groupby("scale_label")
    .agg(**agg_dict)
    .reset_index()
)

display(summary_df)

# ---------------------------------------------------------
# 4) Secondary winners table
# ---------------------------------------------------------
secondary_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    secondary_cols.insert(1, "run_phase")

secondary_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "secondary_affected"
][secondary_cols].sort_values(["scale_label", "official_id"])

display(secondary_df)

# ---------------------------------------------------------
# 5) Control winners table
# ---------------------------------------------------------
control_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
    "activated_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    control_cols.insert(1, "run_phase")

control_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "control"
][control_cols].sort_values(["scale_label", "official_id"])

display(control_df)

# ---------------------------------------------------------
# 6) Optional extra: best-by-scale overview
# ---------------------------------------------------------
best_by_scale_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_group",
    "best_design_pattern",
    "best_p95_ms",
]

best_by_scale_df = cross_scale_schemalens_df[best_by_scale_cols].sort_values(
    ["scale_label", "official_id"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 7) Save outputs
# ---------------------------------------------------------
out_dir = base_results / "cross_scale_analysis_final"
out_dir.mkdir(parents=True, exist_ok=True)

cross_scale_schemalens_df.to_csv(out_dir / "imdb_cross_scale_schemalens_analysis_final.csv", index=False)
summary_df.to_csv(out_dir / "imdb_cross_scale_summary_final.csv", index=False)
secondary_df.to_csv(out_dir / "imdb_cross_scale_secondary_winners_final.csv", index=False)
control_df.to_csv(out_dir / "imdb_cross_scale_control_winners_final.csv", index=False)
best_by_scale_df.to_csv(out_dir / "imdb_cross_scale_best_by_scale_final.csv", index=False)

# Optional extra near-best table
if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    near_best_summary_df = (
        cross_scale_schemalens_df
        .groupby("scale_label")
        .agg(
            near_best_preservation=("near_best_preserved_by_activated", "mean"),
            mean_activated_regret=("activated_regret", "mean"),
        )
        .reset_index()
    )
    near_best_summary_df.to_csv(out_dir / "imdb_cross_scale_near_best_summary_final.csv", index=False)
    display(near_best_summary_df)

print("Saved to:", out_dir)
